# 12 — Pure local semantic retrieval

Iteration 11: all-MiniLM-L6-v2 creates normalized 384-dimensional vectors; FAISS IndexFlatIP searches them exactly by inner product (equivalent to cosine similarity after normalization). There is no BM25 routing, score fusion, LLM, or gold source filter.

The index searches all selected contracts using the original full query. Passage chunks, original text/offsets, and k values match the existing setup. The model is pinned and loads from local files. Model card: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 .

This notebook reviews the saved train run and tries a query. It does not run another experiment or evaluate test data.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
if ROOT.name=='notebooks':
    ROOT=ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
from improve import read_log,read_json
from search_service import build_model,search,EXAMPLE_QUERY

runs=[r for r in read_log(ROOT/'experiments/semantic_log.jsonl') if r['iteration']==11]
assert runs,'Complete python manage.py evaluate_dense first.'
run=runs[-1]
rows=[]
for label,result in run['metrics'].items():
    m=next(m for m in result['retrieval'] if m['k']==5)
    rows.append(dict(approach=label,precision=m['precision'],recall=m['recall'],
                     benchmark_precision=m['benchmark_precision'],benchmark_recall=m['benchmark_recall'],
                     false_abstention=result['false_abstention'],correct_abstention=result['correct_abstention']))
display(pd.DataFrame(rows))
print('Accepted replacement:',run['decision']['accepted'])
display(pd.Series(run['decision']['checks'],name='Passed'))
print('Before:',run['diagnosis_before'])
print('After:',run['diagnosis_after'])

,approach,precision,recall,benchmark_precision,benchmark_recall,false_abstention,correct_abstention
0,baseline,0.017433,0.181647,0.017594,0.192427,0.0,0.0
1,reference,0.078798,0.670075,0.083766,0.765975,0.0,0.0
2,candidate,0.040341,0.357489,0.041892,0.389043,0.0,0.0


Accepted replacement: False


recall_gain                False
precision                  False
answerable_abstention       True
unanswerable_abstention     True
latency                     True
Name: Passed, dtype: bool

Before: {'complete_coverage': 82, 'right_document_wrong_section': 36, 'partial_coverage': 22}
After: {'complete_coverage': 33, 'right_document_wrong_section': 75, 'partial_coverage': 32}


## Check truncation and timing

The basic MiniLM model defaults to 256 wordpieces. We intentionally kept that published default and count inputs exceeding it. Original chunks can be longer than their embedded representation; retrieval may therefore select a passage whose tail was not encoded. This is a documented baseline limitation, not hidden preprocessing.

Index/model building is separate from query search. Saved vectors allow replay without re-encoding known texts. The operational decision uses estimated first-use query encoding plus vector-search time, not just a fast vector-cache read.

In [3]:
print('Config:',run['candidate_config'])
print('Build seconds:',run['build_seconds'])
print('Document inputs above limit:',run['document_inputs_over_token_limit'])
print('Query inputs above limit:',run['query_inputs_over_token_limit'])
print('Vector cache:',run['vector_cache_directory'])
print('Estimated first-use query latency (ms):',run['estimated_first_use_latency_ms'])
print('Observed query latency (ms):',run['metrics']['candidate']['latency_ms'])
print('Initial query cache use:',run['cache_hits'])

Config: {'name': 'dense_minilm_faiss', 'retriever_type': 'dense', 'model_id': 'sentence-transformers/all-MiniLM-L6-v2', 'revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'dimension': 384, 'max_seq_length': 256, 'device': 'cpu', 'torch_threads': 4, 'batch_size': 32, 'chunk_size': 1000, 'chunk_overlap': 200, 'min_score': 0.0, 'k_values': [1, 3, 5, 10], 'query_mode': 'full', 'normalize_embeddings': True, 'faiss_index': 'IndexFlatIP'}
Build seconds: 76.511675799964
Document inputs above limit: 105
Query inputs above limit: 0
Vector cache: cache/dense/06c23934081c90b590f6c0ba128cb03e127c30a91f0b9609ae670ceb568a7a26
Estimated first-use query latency (ms): {'median': 10.17034996766597, 'p95': 12.974840047536416}
Observed query latency (ms): {'median': 29.60669994354248, 'p95': 75.78495498164537}
Initial query cache use: {'miss': 163}


## Try the dense model

The configured model encodes query meaning and compares it with passage vectors. FAISS is the search engine, not the embedding model. Scores are cosine similarities, not answer probabilities. This model searches globally; do not describe it as using the accepted BM25 document router.

In [4]:
configuration='dense_minilm_faiss'
question=EXAMPLE_QUERY
model=build_model(configuration,ROOT)
result=search(model,question,k=5)
print(result['configuration'],result['message'])
if result['hits']:
    display(pd.DataFrame(result['hits'])[['score','file_path','start','end','text']])

dense_minilm_faiss 5 passages retrieved.


,score,file_path,start,end,text
0,0.672655,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,0.547689,cuad/HOLIDAYRVSUPERSTORESINC_04_15_2002-EX-10....,4800,5800,"he License (the ""Endorsement"").\n\n 3. ..."
2,0.541084,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
3,0.514982,cuad/PcquoteComInc_19990721_S-1A_EX-10.11_6377...,4000,5000,ll terminate or alter this Agreement except as...
4,0.497287,cuad/PcquoteComInc_19990721_S-1A_EX-10.11_6377...,3200,4200,ABW agrees to include the following in the co...


## Explain this step

“We added a separate semantic baseline to test whether vector similarity can retrieve relevant clauses without keyword routing. We fixed the corpus, chunks, offsets, and scoring, cached model-specific vectors, and measured train quality and abstention. The experiment is retained even if it does not beat our tuned BM25 pipeline.”

A first dense baseline is not evidence that semantic retrieval is optimal. General-purpose training, truncation, full-query phrasing, document selection, and an untuned positive-cosine threshold can all matter. Keep these limitations and failed guardrails visible.

Kanon comparison is planned next and requires separate provider access and review of any public corpus passages sent for embedding. The selected dense setup plus Gemini follows after that. Hybrid/reranker work remains deferred until submission readiness. Test performance remains unmeasured.

On this Windows setup, notebook/web embedding requests run in a small local worker process to avoid a PyTorch DLL initialization issue inside those runtimes. The pinned model and vector calculations are unchanged; known-query vectors replay directly from cache.